In [1]:
import pandas as pd
import numpy as np

## データの読み込み

In [2]:
url = "https://gksmyth.github.io/ozdasl/oz/stroke.txt"
df = pd.read_csv(url, sep="\t")  # 区切りがタブっぽいので \t を指定

In [3]:
df

,Subject,Group,Sex,Side,Age,Lapse,UE1,UE2,UE3,UE4,...,Bal7,Bal8,Bart1,Bart2,Bart3,Bart4,Bart5,Bart6,Bart7,Bart8
0,I,E,M,R,74,11.00,15,23,23,26,...,10,10,45,45,45,45,80,80,80,90
1,II,E,M,L,61,2.00,3,3,3,5,...,8,8,20,25,25,25,30,35,30,50
2,III,E,M,R,49,6.00,2,2,4,7,...,10,10,50,50,55,70,70,75,90,90
3,IV,E,M,L,68,1.00,14,25,25,31,...,9,9,25,25,35,40,60,60,70,80
4,V,E,M,R,55,33.00,22,22,22,22,...,12,12,100,100,100,100,100,100,100,100
5,VI,E,M,L,64,0.29,5,14,16,26,...,9,9,20,20,30,50,50,60,85,95
6,VII,E,M,L,59,5.00,1,3,4,5,...,9,9,30,35,35,40,50,60,75,85
7,VIII,E,M,R,62,0.43,2,2,3,4,...,10,10,30,35,45,50,55,65,65,70
8,1,F,M,R,58,1.00,4,4,4,4,...,11,11,40,55,60,70,80,85,90,90
9,2,F,M,R,41,4.00,17,22,22,22,...,12,12,65,65,70,70,80,80,80,80


## 分析用データに加工

In [4]:
# 使用項目の選定
usecols = [
    "Subject",
    "Group",
    "Bart1",
    "Bart2",
    "Bart3",
    "Bart4",
    "Bart5",
    "Bart6",
    "Bart7",
    "Bart8",
]
df = df[usecols]

# Groupをダミー変数にする（alpha）
dummy = pd.get_dummies(df["Group"], columns=["Group"]).astype(int)
dummy = dummy.loc[np.repeat(dummy.index, 8)].reset_index(drop=True)

# 分散分析しやすいようにデータ加工
X = (
    df.drop(["Subject", "Group"], axis=1)
    .stack()
    .reset_index()
    .drop(["level_0"], axis=1)
    .rename(columns={"level_1": "week", 0: "Berthel_Score"})
)
X = pd.concat([dummy, X], axis=1)

# Groupをダミー変数にする（alpha）
dummy = pd.get_dummies(df["Group"], columns=["Group"]).astype(int)
dummy = dummy.loc[np.repeat(dummy.index, 8)].reset_index(drop=True)

# 分散分析しやすいようにデータ加工
X = (
    df.drop(["Subject", "Group"], axis=1)
    .stack()
    .reset_index()
    .drop(["level_0"], axis=1)
    .rename(columns={"level_1": "week", 0: "Berthel_Score"})
)
X = pd.concat([dummy, X], axis=1)

# weekはそのまま数字を入れる（beta）
X["week"] = X["week"].str[4:].astype(int)
X["beta1_week"] = X["E"] * X["week"]
X["beta2_week"] = X["F"] * X["week"]
X["beta3_week"] = X["G"] * X["week"]

# 推定対象はalpha2, alpha3ではなく、alpha2 - alpha1, alpha3 - alpha1 とみなすので、
# 基準のとなるEはすべて１とする
# beta1, beta2, beta3 についても同様
X["E"] = 1
X["beta1_week"] = [1, 2, 3, 4, 5, 6, 7, 8] * 24

# 目的変数と説明変数に分ける
y = X["Berthel_Score"]
X["ids"] = np.repeat(np.arange(24), 8)
# X = X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values

In [8]:
X[:10]

,E,F,G,week,Berthel_Score,beta1_week,beta2_week,beta3_week,ids
0,1,0,0,1,45,1,0,0,0
1,1,0,0,2,45,2,0,0,0
2,1,0,0,3,45,3,0,0,0
3,1,0,0,4,45,4,0,0,0
4,1,0,0,5,80,5,0,0,0
5,1,0,0,6,80,6,0,0,0
6,1,0,0,7,80,7,0,0,0
7,1,0,0,8,90,8,0,0,0
8,1,0,0,1,20,1,0,0,1
9,1,0,0,2,25,2,0,0,1


In [6]:
X["ids"].nunique()

24

## p237の方法により最尤推定量を数値計算する

In [7]:
np.set_printoptions(suppress=True, precision=3)

### 相関構造が独立の場合

In [10]:
V_hat = np.array([np.eye(8) for j in range(24)])

for n_iter in range(10):
    list_first_term = []
    list_second_term = []

    for i in range(X["ids"].nunique()):
        V_i = V_hat[i]

        X_i = X[X["ids"] == i]
        y_i = X_i["Berthel_Score"].values

        X_i = X_i[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values

        list_first_term.append((X_i.T) @ (np.linalg.inv(V_i)) @ (X_i))
        list_second_term.append((X_i.T) @ (np.linalg.inv(V_i)) @ (y_i))

    list_first_term = np.array(list_first_term)
    list_second_term = np.array(list_second_term)

    # 第2項を計算
    mat_second_term = list_second_term.sum(axis=0)

    # 第1項を計算
    mat_first_term = list_first_term.sum(axis=0)

    # 係数を推定
    beta_hat = np.linalg.inv(mat_first_term) @ mat_second_term

    # 線形予測子を計算
    mu_hat = (
        X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values @ beta_hat
    )

    # 残差を計算
    r = y.values - mu_hat
    # r = r.reshape(24, 8)

    # # 残差の分散共分散行列を計算
    V_hat = (r.reshape((192, 1)) @ r.reshape((192, 1)).T) / (X.shape[0] - 6)
    V_hat = np.array(
        [V_hat[8 * i : 8 * (i + 1), 8 * i : 8 * (i + 1)] for i in range(24)]
    )
    # V_hat = np.array([(r[i].reshape((8,1)) @ r[i].reshape((8,1)).T) / (X.shape[0] - 6) for i in range(len(r))])
    print(beta_hat)

[29.821  3.348 -0.022  6.324 -1.994 -2.686]


LinAlgError: Singular matrix

### 相関構造が等相関の場合

In [9]:
import numpy as np


class GEE:
    def __init__(self, family="binomial", link="logit", max_iter=100, tol=1e-11):
        self.family = family
        self.link = link
        self.max_iter = max_iter
        self.tol = tol
        self.beta_ = None
        self.rho_ = 0.0  # exchangeable 相関係数

    def _inv_link(self, eta):
        if self.link == "logit":
            return 1 / (1 + np.exp(-eta))
        elif self.link == "identity":
            return eta
        else:
            raise NotImplementedError

    def _variance(self, mu):
        if self.family == "binomial":
            return mu * (1 - mu)
        elif self.family == "gaussian":
            return np.ones_like(mu)
        else:
            raise NotImplementedError

    def _estimate_rho(self, groups, resid, var):
        num = 0.0
        den = 0.0
        for g in np.unique(groups):
            idx = groups == g
            r = resid[idx] / np.sqrt(var[idx])
            n = len(r)
            if n > 1:
                for j in range(n):
                    for k in range(j + 1, n):
                        num += r[j] * r[k]
                den += n * (n - 1) / 2
        return num / den if den > 0 else 0.0

    def fit(self, X, y, groups):
        X = np.asarray(X)
        y = np.asarray(y)
        groups = np.asarray(groups)
        n_features = X.shape[1]

        beta = np.zeros(n_features)
        rho = 0.0
        psi = 1

        for it in range(self.max_iter):
            U = np.zeros(n_features)
            H = np.zeros((n_features, n_features))

            all_mu = np.zeros_like(y, dtype=float)
            all_var = np.zeros_like(y, dtype=float)
            all_resid = np.zeros_like(y, dtype=float)

            for g in np.unique(groups):
                idx = groups == g
                Xg = X[idx]
                yg = y[idx]

                eta = Xg @ beta
                mu = self._inv_link(eta)
                var = self._variance(mu)

                all_mu[idx] = mu
                all_var[idx] = var
                all_resid[idx] = yg - mu
                A = np.diag(var)

                # exchangeable correlation matrix
                n = len(mu)
                R = (1 - rho) * np.eye(n) + rho * np.ones((n, n))
                Vi = np.sqrt(A) @ R @ np.sqrt(A) * psi

                # D_i = ∂μ/∂β
                if self.link == "logit":
                    D = (mu * (1 - mu))[:, None] * Xg
                else:
                    D = Xg
                Vinv = np.linalg.inv(Vi)

                U += D.T @ Vinv @ (yg - mu)
                H += D.T @ Vinv @ D

            step = np.linalg.solve(H, U)
            beta_new = beta + step

            # rho を更新（残差ベース）
            rho_new = self._estimate_rho(groups, all_resid, all_var)
            if (
                np.linalg.norm(beta_new - beta) < self.tol
                and abs(rho_new - rho) < self.tol
            ):
                beta, rho = beta_new, rho_new
                break

            beta, rho = beta_new, rho_new

        # --- サンドイッチ分散推定量 ---
        H = np.zeros((n_features, n_features))
        M = np.zeros((n_features, n_features))

        for g in np.unique(groups):
            idx = groups == g
            Xg = X[idx]
            yg = y[idx]

            eta = Xg @ beta
            mu = self._inv_link(eta)
            var = self._variance(mu)
            resid = yg - mu

            A = np.diag(var)
            n = len(mu)
            R = (1 - rho) * np.eye(n) + rho * np.ones((n, n))
            Vi = np.sqrt(A) @ R @ np.sqrt(A)

            if self.link == "logit":
                D = (mu * (1 - mu))[:, None] * Xg
            else:
                D = Xg

            Vinv = np.linalg.inv(Vi)

            H += D.T @ Vinv @ D
            M += D.T @ Vinv @ np.outer(resid, resid) @ Vinv @ D

        cov_beta = np.linalg.inv(H) @ M @ np.linalg.inv(H)
        se_beta = np.sqrt(np.diag(cov_beta))

        self.beta_ = beta
        self.rho_ = rho
        self.cov_beta_ = cov_beta
        self.se_beta_ = se_beta

        return self

In [10]:
gee = GEE(family="gaussian", link="identity", max_iter=300)
gee.fit(
    X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values,
    X["Berthel_Score"],
    X["ids"].values,
)
print("推定された β:", gee.beta_)
print("推定された ρ:", gee.rho_)
print("各係数の標準誤差:", gee.se_beta_)

推定された β: [29.821  3.348 -0.022  6.324 -1.994 -2.686]
推定された ρ: 353.52556224312633
各係数の標準誤差: [10.176 11.633 10.896  1.131  1.478  1.47 ]
